对vive_tracker与sandbox进行刚体转换的类

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

class Point:
    """
    表示三维空间中的一个点，包含位置和方向信息。
    """
    def __init__(self, position=None, orientation=None):
        """
        初始化Point对象。
        :param position: 点的位置，3D向量 [x, y, z]，可以为None
        :param orientation: 点的方向，欧拉角 [roll,yaw,pitch]，可以为None
        """
        if position is not None:
            self.position = np.array(position)
        else:
            self.position = None
        if orientation is not None:
            self.orientation = np.array(orientation)
        else:
            self.orientation = None

class CoordinateTransformer:
    """
    用于计算和应用坐标变换的类。
    """
    def __init__(self, orientations_A=None, orientations_B=None, positions_A=None, positions_B=None):
        """
        初始化CoordinateTransformer对象。
        T_pos: 位置变换矩阵
        R_euler: 方向变换矩阵
        """
        self.T_pos = None
        self.R_euler = None

        # 存储输入数据
        self.__orientations_A = orientations_A
        self.__orientations_B = orientations_B
        self.__positions_A = positions_A
        self.__positions_B = positions_B

        # 如果初始化时提供了数据，则直接计算变换矩阵
        if orientations_A is not None and orientations_B is not None:
            self.calculate_orientation_transformation(orientations_A, orientations_B)
        if positions_A is not None and positions_B is not None:
            self.calculate_position_transformation(positions_A, positions_B)

    @staticmethod
    def euler_to_matrix(euler):
        """
        将欧拉角 (yxz) 转换为旋转矩阵。
        """
        return R.from_euler('yxz', [euler[2], euler[0], euler[1]]).as_matrix()

    @staticmethod
    def matrix_to_euler(matrix):
        """
        将旋转矩阵转换为欧拉角 (yxz)。
        返回: [yaw, pitch, roll] 顺序的欧拉角（弧度）。
        """
        euler = R.from_matrix(matrix).as_euler('yxz')
        return [euler[0], euler[2], euler[1]]

    @staticmethod
    def construct_transformation_matrix(rotation_matrix, translation):
        """
        根据旋转矩阵和平移向量构建变换矩阵。
        """
        T = np.eye(4)
        T[:3, :3] = rotation_matrix
        T[:3, 3] = translation
        return T

    def calculate_position_transformation(self, positions_A=None, positions_B=None):
        """
        计算两组点之间的位置变换矩阵。
        返回T_pos与R_pos
        """
        if positions_A is not None and positions_B is not None:
            self.__positions_A = positions_A
            self.__positions_B = positions_B
        elif self.__positions_A is None or self.__positions_B is None:
            print("请先提供位置数据！")
            return

        # 使用存储的数据进行计算
        centroid_A = np.mean(self.__positions_A, axis=0)
        centroid_B = np.mean(self.__positions_B, axis=0)

        H = np.dot((self.__positions_A - centroid_A).T, (self.__positions_B - centroid_B))
        U, S, Vt = np.linalg.svd(H)
        R_pos = np.dot(Vt.T, U.T)
        if np.linalg.det(R_pos) < 0:
            Vt[-1, :] *= -1
            R_pos = np.dot(Vt.T, U.T)
        translation = centroid_B.T - np.dot(R_pos, centroid_A.T)
        self.T_pos = self.construct_transformation_matrix(R_pos, translation)
        return R_pos, translation

    def calculate_orientation_transformation(self, orientations_A=None, orientations_B=None):
        """
        计算两组方向之间的方向变换矩阵。
        """
        if orientations_A is not None and orientations_B is not None:
            self.__orientations_A = orientations_A
            self.__orientations_B = orientations_B
        elif self.__orientations_A is None or self.__orientations_B is None:
            print("请先提供方向数据！")
            return

        # 使用存储的数据进行计算
        R_A = [self.euler_to_matrix(orientation) for orientation in self.__orientations_A]
        R_B = [self.euler_to_matrix(orientation) for orientation in self.__orientations_B]

        R_diff = [np.dot(R_B[i], R_A[i].T) for i in range(len(R_A))]
        R_mean = np.mean(R_diff, axis=0)
        U, _, Vt = np.linalg.svd(R_mean)
        self.R_euler = np.dot(U, Vt)

    def apply_position_transformation(self, position):
        """
        应用位置变换到给定点。
        """
        position_homogeneous = np.append(position, 1)
        transformed_position_homogeneous = np.dot(self.T_pos, position_homogeneous)
        return transformed_position_homogeneous[:3]

    def apply_orientation_transformation(self, orientation):
        """
        应用方向变换到给定方向。
        """
        R_point = self.euler_to_matrix(orientation)
        R_transformed = np.dot(self.R_euler, R_point)
        transformed_orientation = self.matrix_to_euler(R_transformed)
        return transformed_orientation

    def transform_point(self, point):
        """
        对给定点应用完整的变换（位置和方向）。
        """
        transformed_position = self.apply_position_transformation(point.position)
        transformed_orientation = self.apply_orientation_transformation(point.orientation)
        return Point(transformed_position, transformed_orientation)

    def calculate_position_error(self, actual, predicted):
        """
        计算位置的均方根误差（RMSE）
        """
        return np.sqrt(np.mean(np.sum((np.array(actual) - np.array(predicted))**2, axis=1)))

    def calculate_orientation_error(self, actual, predicted):
        """
        计算方向的平均角度误差（度）
        输入和输出都是以弧度为单位
        """
        actual = np.array(actual)
        predicted = np.array(predicted)

        # 计算每个轴的角度差
        diff = np.abs(actual - predicted)

        # 处理角度差大于π的情况
        diff = np.where(diff > np.pi, 2 * np.pi - diff, diff)

        # 计算平均角度误差（度）
        mean_error_deg = np.rad2deg(np.mean(diff))  # 将弧度转换为度

        return mean_error_deg

    def calculate_and_print_errors(self, positions_B=None, orientations_B=None, transformed_positions=None, transformed_orientations=None):
        """
        计算并打印位置和方向的误差
        """
        if positions_B is not None and transformed_positions is not None:
            position_error = self.calculate_position_error(positions_B, transformed_positions)
            print(f"位置均方根误差 (RMSE): {position_error:.4f} 米")
            # 打印每个点的位置误差
            for i in range(len(positions_B)):
                pos_error = np.linalg.norm(np.array(positions_B[i]) - np.array(transformed_positions[i]))
                print(f"点 {i} 的位置误差: {pos_error:.4f} 米")

        if orientations_B is not None and transformed_orientations is not None:
            orientation_error = self.calculate_orientation_error(orientations_B, transformed_orientations)
            print(f"方向平均角度误差: {orientation_error:.4f} 度")
            # 打印每个点的方向误差
            for i in range(len(orientations_B)):
                ori_error = self.calculate_orientation_error([orientations_B[i]], [transformed_orientations[i]])
                print(f"点 {i} 的方向误差: {ori_error:.4f} 度")

    def demo_transformation(self):
        """
        演示坐标变换过程，包括创建坐标系、计算变换和验证结果。
        """
        if self.__orientations_A is None or self.__orientations_B is None:
            print("请先提供方向数据！")
            return
        if self.__positions_A is None or self.__positions_B is None:
            print("请先提供位置数据！")
            return

        # 计算变换
        self.calculate_position_transformation()
        self.calculate_orientation_transformation()

        # 打印变换矩阵
        print("位置变换矩阵:")
        print(self.T_pos)

        print("方向变换矩阵:")
        print(self.R_euler)

        # 应用变换
        transformed_positions = [self.apply_position_transformation(position) for position in self.__positions_A]
        transformed_orientations = [self.apply_orientation_transformation(orientation) for orientation in self.__orientations_A]

        # 验证和打印结果
        print("验证转换结果:")
        for i in range(len(self.__positions_A)):
            print(f"点A {i} 在坐标系B中的实际位置: {self.__positions_B[i]}")
            print(f"点A {i} 在坐标系B中的转换位置: {transformed_positions[i]}")
        for i in range(len(self.__orientations_A)):
            print(f"点A {i} 在坐标系B中的实际欧拉角: {np.rad2deg(self.__orientations_B[i])}")
            print(f"点A {i} 在坐标系B中的转换欧拉角: {np.rad2deg(transformed_orientations[i])}")
            print()

        # 计算和打印误差
        self.calculate_and_print_errors(self.__positions_B, self.__orientations_B, transformed_positions, transformed_orientations)

    def get_transformation_matrix(self):
        """
        返回变换矩阵
        以 T_pos,R_euler的顺序返回两个变换矩阵
        """
        return self.T_pos, self.R_euler

A点集是tracker在steamvr中的坐标，B点集是tracker在SandBox中的坐标

在转换的过程中，只使用x,z 与 yaw

In [ ]:
#here we set all y as 0

positions_A = [
    # 8 points
    # 1
    [0.48184, 0, -3.93924],
    # 2
    [0.08725, 0, -3.3454],
    # 3
    [-1.4261, 0, -2.9962],
    # 4
    [-1.83450, -0, -3.27553],
    # 5
    [-0.5324, 0, -2.4248],
    # 8
    [0.8592, 0, -1.4793],
    # a
    [1.0574, 0, -1.7693],
    # b
    [1.2512, 0, -2.0666],
    # c##############################
    #[0.7020, 0, -1.2212],
    # d,
    [0.4604, 0, -0.8474],

    ### 4 matrix
    # 1
    [-1.1320, 0, -4.3054],
    # 2
    [-1.4142, 0, -3.84485],
    # 3
    [-0.9778, 0, -3.6239],
    # 4
    [-0.7041, 0, -4.0413],
    # 1'
    [-2.2357, 0, -2.5922],
    # 2'
    [-2.50755, 0, -2.1746],
    # 3'
    [-2.0894, 0, -1.90105],
    # 4'
    [-1.8161, 0, -2.3207],
    # 1''
    [-0.0468, 0, -1.1820],
    # 2''
    [-0.3184, 0, -0.7581],
    # 3''
    [0.0939, 0, -0.4933],
    # 4''
    [0.3747, 0, -0.9140],
    # 1'''
    [1.0288, 0, -2.8581],
    # 2'''
    [0.7589, 0, -2.4383],
    # 3'''
    [1.1871, 0, -2.1691],
    # 4'''
    [1.4569, 0, -2.5930],

    # 11
    [0.0973, 0, -4.1960],
    # 22
    [-0.2736, 0, -4.4384],
    # 33
    [0.8952, 0, -3.6727],
    # 44
    [1.2936, 0, -3.4120],
]

positions_B = [
    [0.217, 0, -2.05],    #1
    [0.92,  0, -2.05],    #2
    [2.05,  0, -0.97],    #3
    [2.05,  0, -0.49],    #4
    [2.05,  0, -2.05],    #5
    [2.05,  0, -3.72],    #8
    #a
    [1.7,0,-3.726],
    #b
    [1.343,0,-3.723],
    #c#####################
    #[2.473,0,-3.733],
    #d
    [2.795,0,-3.738],

    [0.80,  0, -0.503],   #8 33
    [1.30,  0, -0.502],   #9 34
    [1.30,  0, -1.001],   #10 35
    [0.80,  0, -1.001],   #11 36

    [2.80,  0, -0.50],    #12 37
    [3.30,  0, -0.50],    #13 38
    [3.30,  0, -1.00],    #14 39
    [2.80,  0, -1.00],    #15 40

    [2.80,  0, -3.10],    #16 41
    [3.30,  0, -3.10],    # 42
    [3.30,  0, -3.60],    # 43
    [2.80,  0, -3.60],    # 44

    [0.815, 0, -3.10],   # 45
    [1.316, 0, -3.10],   # 46
    [1.310, 0, -3.60],   # 47
    [0.808, 0, -3.60],   # 48

    ##11
    [0.199,   0,    -1.595],
    ##22
    [0.199,   0,    -1.153],
    ##33
    [0.209,   0,    -2.547],
    ##44
    [0.209,   0,    -3.023]


]


In [ ]:
orientations_A= [
    [np.radians(0), np.radians(-55.4875), np.radians(0)],
    [np.radians(0), np.radians(-101.1943), np.radians(0)],
    [np.radians(0), np.radians(-146.1972), np.radians(0)],
    [np.radians(0), np.radians(168.6552), np.radians(0)],
    [np.radians(0), np.radians(123.3208), np.radians(0)],
    [np.radians(0), np.radians(80.7334), np.radians(0)],
    [np.radians(0), np.radians(32.1998), np.radians(0)],
    [np.radians(0), np.radians(-57.5099), np.radians(0)],
    [np.radians(0), np.radians(-54.3468), np.radians(0)],
    [np.radians(0), np.radians(-52.7843), np.radians(0)],
    [np.radians(0), np.radians(-7.1385), np.radians(0)],
    [np.radians(0), np.radians(34.0582), np.radians(0)],
    [np.radians(0), np.radians(74.8816), np.radians(0)],
    [np.radians(0), np.radians(124.6180), np.radians(0)],
    [np.radians(0), np.radians(172.8003), np.radians(0)],
    [np.radians(0), np.radians(-145.1366), np.radians(0)],
    [np.radians(0), np.radians(-95.9682), np.radians(0)],
    [np.radians(0), np.radians(-53.2292), np.radians(0)],
    [np.radians(0), np.radians(-6.7862), np.radians(0)],
    [np.radians(0), np.radians(36.0104), np.radians(0)],
    [np.radians(0), np.radians(-53.0270), np.radians(0)],
    [np.radians(0), np.radians(-12.2344), np.radians(0)],
    [np.radians(0), np.radians(30.9962), np.radians(0)],
    [np.radians(0), np.radians(121.4363), np.radians(0)]
]
orientations_B=  [
    [np.radians(0), np.radians(0), np.radians(0)],
    [np.radians(0), np.radians(-45.7068), np.radians(0)],
    [np.radians(0), np.radians(-90.7097), np.radians(0)],
    [np.radians(0), np.radians(-135.8027), np.radians(0)],
    [np.radians(0), np.radians(178.8677), np.radians(0)],
    [np.radians(0), np.radians(136.2333), np.radians(0)],
    [np.radians(0), np.radians(87.7123), np.radians(0)],
    [np.radians(0), np.radians(-2.0224), np.radians(0)],
    [np.radians(0), np.radians(1.1407), np.radians(0)],
    [np.radians(0), np.radians(2.7032), np.radians(0)],
    [np.radians(0), np.radians(48.349), np.radians(0)],
    [np.radians(0), np.radians(89.5457), np.radians(0)],
    [np.radians(0), np.radians(130.3691), np.radians(0)],
    [np.radians(0), np.radians(180.1055), np.radians(0)],
    [np.radians(0), np.radians(-131.7128), np.radians(0)],
    [np.radians(0), np.radians(-89.6491), np.radians(0)],
    [np.radians(0), np.radians(-40.4807), np.radians(0)],
    [np.radians(0), np.radians(2.2583), np.radians(0)],
    [np.radians(0), np.radians(48.7013), np.radians(0)],
    [np.radians(0), np.radians(91.4979), np.radians(0)],
    [np.radians(0), np.radians(2.4605), np.radians(0)],
    [np.radians(0), np.radians(43.2531), np.radians(0)],
    [np.radians(0), np.radians(86.4837), np.radians(0)],
    [np.radians(0), np.radians(176.9238), np.radians(0)]
]

获得转换矩阵

In [ ]:
transformer = CoordinateTransformer(orientations_A, orientations_B, positions_A, positions_B)

# 获取变换矩阵
T_pos, R_euler = transformer.get_transformation_matrix()
# 打印变换矩阵
#print("位置变换矩阵:")
#print(T_pos)

#print("方向变换矩阵:")
#print(R_euler)

# 使用 demo_transformation 方法演示变换过程
transformer.demo_transformation()


位置变换矩阵:
[[-0.54573909  0.          0.83795516  3.77076165]
 [ 0.          1.          0.          0.        ]
 [-0.83795516  0.         -0.54573909 -3.79733569]
 [ 0.          0.          0.          1.        ]]
方向变换矩阵:
[[ 0.56649565 -0.82406473  0.        ]
 [ 0.82406473  0.56649565  0.        ]
 [ 0.          0.          1.        ]]
验证转换结果:
点A 0 在坐标系B中的实际位置: [0.217, 0, -2.05]
点A 0 在坐标系B中的转换位置: [ 0.20689625  0.         -2.05129876]
点A 1 在坐标系B中的实际位置: [0.92, 0, -2.05]
点A 1 在坐标系B中的转换位置: [ 0.91985072  0.         -2.04473173]
点A 2 在坐标系B中的实际位置: [2.05, 0, -0.97]
点A 2 在坐标系B中的转换位置: [ 2.03835891  0.         -0.96718438]
点A 3 在坐标系B中的实际位置: [2.05, 0, -0.49]
点A 3 在坐标系B中的转换位置: [ 2.02717274  0.         -0.47252219]
点A 4 在坐标系B中的实际位置: [2.05, 0, -2.05]
点A 4 在坐标系B中的转换位置: [ 2.02943947  0.         -2.02790022]
点A 5 在坐标系B中的实际位置: [2.05, 0, -3.72]
点A 5 在坐标系B中的转换位置: [ 2.06227556  0.         -3.70999493]
点A 6 在坐标系B中的实际位置: [1.7, 0, -3.726]
点A 6 在坐标系B中的转换位置: [ 1.71110308  0.         -3.7178133 ]
点A 7 在坐标系B中的实际位

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def transform_point_pose(position, orientation, T_pos, R_euler):
    """
    使用给定的位置变换矩阵 (T_pos) 和方向变换矩阵 (R_euler) 对点的方位和方向进行转换。

    参数:
        position: 点的原始位置，3D 向量 [x, y, z]。
        orientation: 点的原始方向，欧拉角 [roll, pitch, yaw]，单位为弧度。
        T_pos: 位置变换矩阵，4x4 矩阵。
        R_euler: 方向变换矩阵，3x3 矩阵。

    返回值:
        transformed_position: 转换后的位置，3D 向量 [x', y', z']。
        transformed_orientation: 转换后的方向，欧拉角 [roll', pitch', yaw']，单位为弧度。
    """

    # 将位置转换为齐次坐标
    position_homogeneous = np.append(position, 1)

    # 应用位置变换
    transformed_position_homogeneous = T_pos @ position_homogeneous
    transformed_position = transformed_position_homogeneous[:3]

    # 将欧拉角转换为旋转矩阵
    R_point = R.from_euler('yxz', [orientation[2], orientation[0], orientation[1]]).as_matrix()

    # 应用方向变换
    R_transformed = R_euler @ R_point

    # 将旋转矩阵转换回欧拉角
    transformed_orientation = R.from_matrix(R_transformed).as_euler('yxz')

    # 调整欧拉角顺序
    transformed_orientation = [transformed_orientation[0], transformed_orientation[2], transformed_orientation[1]]

    return transformed_position, transformed_orientation

In [ ]:
[-0.5236, -1.3480, -2.4197][ 0.7017, -56.5258, 90.4932]

验证旋转转换

In [ ]:
# 假设你已经获得了 T_pos 和 R_euler 矩阵

# 定义点的原始方位和方向
position = [0.7333,0,-3.7766],  # 例如
#orientation = [ np.radians(0), np.radians(180.5258),np.radians( 0)]
orientations=[]
for i in range(-180,180):
  orientations.append([np.radians(0), np.radians(i),np.radians( 0)])

for orientation in orientations:
  print("转换前的方向 (弧度):", orientation)
  print("转换前的方向 (角度):", np.rad2deg(orientation))
  transformed_position, transformed_orientation = transform_point_pose(position, orientation, T_pos, R_euler)
  #print("转换后的位置:", transformed_position)
  print("转换后的方向 (弧度):", transformed_orientation)
  print("转换后的方向 (角度):", np.rad2deg(transformed_orientation))
# 调用函数进行转换


# 打印结果


转换前的方向 (弧度): [0.0, -3.141592653589793, 0.0]
转换前的方向 (角度): [   0. -180.    0.]
转换后的方向 (弧度): [0.0, -2.173043420044534, 0.0]
转换后的方向 (角度): [   0.         -124.50621667    0.        ]
转换前的方向 (弧度): [0.0, -3.12413936106985, 0.0]
转换前的方向 (角度): [   0. -179.    0.]
转换后的方向 (弧度): [0.0, -2.155590127524591, 0.0]
转换后的方向 (角度): [   0.         -123.50621667    0.        ]
转换前的方向 (弧度): [0.0, -3.1066860685499065, 0.0]
转换前的方向 (角度): [   0. -178.    0.]
转换后的方向 (弧度): [0.0, -2.138136835004648, 0.0]
转换后的方向 (角度): [   0.         -122.50621667    0.        ]
转换前的方向 (弧度): [0.0, -3.0892327760299634, 0.0]
转换前的方向 (角度): [   0. -177.    0.]
转换后的方向 (弧度): [0.0, -2.120683542484705, 0.0]
转换后的方向 (角度): [   0.         -121.50621667    0.        ]
转换前的方向 (弧度): [0.0, -3.07177948351002, 0.0]
转换前的方向 (角度): [   0. -176.    0.]
转换后的方向 (弧度): [0.0, -2.103230249964761, 0.0]
转换后的方向 (角度): [   0.         -120.50621667    0.        ]
转换前的方向 (弧度): [0.0, -3.0543261909900767, 0.0]
转换前的方向 (角度): [   0. -175.    0.]
转换后的方向 (弧度): [0.0, -2.0857769574